# Fetal ECG Analysis and Extraction
Pipeline implementation based on *A robust fetal ECG detection method for abdominal recordings* (Martens et al., 2007).

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import wfdb
from sklearn.decomposition import FastICA

from filtering import BaselineWanderRemover, PowerLineCanceller
from preprocessing import SignalResampler
from mecg_canceller import MECGCanceller
from fecg_extractor import FECGExtractor

# Setup plot style
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Data Loading (WFDB)
We load a real record from the local `data/` folder.

In [ ]:
# Loading 'ecg01' from the local data directory
data_path = '../data/ecg01'
try:
    record = wfdb.rdrecord(data_path)
    fs_original = record.fs
    # Transpose to get shape: (n_channels, n_samples)
    channels = record.p_signal.T 
    print(f"Loaded {channels.shape[0]} channels with {channels.shape[1]} samples at {fs_original} Hz.")
except Exception as e:
    print(f"Error loading data from {data_path}. Ensure the files are present in the data/ folder.")
    print(e)

## 2. Pipeline Execution & Visualization

In [ ]:
plot_samples = int(fs_original * 3) # Plot 3 seconds
t_orig = np.arange(plot_samples) / fs_original

# 1. Baseline Wander Removal
bwr = BaselineWanderRemover(fs=fs_original)
filtered_bwr = bwr.apply(channels)

# Plot Step 1
plt.figure(figsize=(12, 4))
plt.plot(t_orig, channels[0, :plot_samples], label='Raw', alpha=0.6)
plt.plot(t_orig, filtered_bwr[0, :plot_samples], label='Baseline Removed', linewidth=1.5)
plt.title('Baseline Wander Removal (Channel 0)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

In [ ]:
# 2. Power-Line Canceller
plc = PowerLineCanceller(fs=fs_original)
filtered_plc = np.zeros_like(filtered_bwr)
for ch_idx in range(filtered_bwr.shape[0]):
    filtered_plc[ch_idx, :] = plc.apply(filtered_bwr[ch_idx, :])

# Plot Step 2
plt.figure(figsize=(12, 4))
plt.plot(t_orig, filtered_bwr[0, :plot_samples], label='Before PLC', alpha=0.6)
plt.plot(t_orig, filtered_plc[0, :plot_samples], label='After PLC', linewidth=1.5)
plt.title('Power-Line Interference Cancellation (Channel 0)')
plt.xlabel('Time (s)')
plt.legend()
plt.show()

In [ ]:
# 3. Upsampling
fs_new = 2000.0
resampler = SignalResampler(original_fs=fs_original, target_fs=fs_new)
upsampled_signal = resampler.upsample(filtered_plc)
print(f"Signal upsampled to {fs_new} Hz. New shape: {upsampled_signal.shape}")

In [ ]:
# 4. MECG Canceller
mecg_canc = MECGCanceller(fs=fs_new)
pc1 = mecg_canc.extract_principal_component(upsampled_signal)
maternal_peaks = mecg_canc.detect_qrs_matched_filter(pc1)
residual = mecg_canc.subtract_mecg_least_squares(pc1, maternal_peaks)

plot_samples_up = int(fs_new * 3)
t_up = np.arange(plot_samples_up) / fs_new

# Plot Step 4
plt.figure(figsize=(12, 4))
plt.plot(t_up, pc1[:plot_samples_up], label='Maternal PC1', alpha=0.6)
plt.plot(t_up, residual[:plot_samples_up], label='Residual (Fetal + Noise)', linewidth=1.5)
plt.title('Maternal ECG Cancellation (Standard Least Squares)')
plt.xlabel('Time (s)')
plt.legend()
plt.show()

In [ ]:
# 5. FECG Extractor
fecg_ext = FECGExtractor(fs=fs_new)
fetal_peaks = fecg_ext.detect_fetal_qrs(residual, template_width_sec=0.08) # Adjusted for real data

if len(fetal_peaks) > 1:
    fhr = fecg_ext.compute_fhr(fetal_peaks)
    fecg_template = fecg_ext.synchronous_averaging(residual, fetal_peaks)
    
    print(f"Mean Fetal Heart Rate (FHR): {np.mean(fhr):.2f} bpm")
    
    plt.figure(figsize=(6, 4))
    if fecg_template is not None:
        plt.plot(np.linspace(-200, 200, len(fecg_template)), fecg_template, color='red')
        plt.title('Synchronously Averaged Fetal ECG Complex')
        plt.xlabel('Time (ms)')
        plt.ylabel('Amplitude')
        plt.show()
else:
    print("Could not detect enough fetal peaks.")

## 3. Validation: Power Spectral Density (Welch)

In [ ]:
f_raw, Pxx_raw = signal.welch(channels[0], fs=fs_original, nperseg=2048)
f_filt, Pxx_filt = signal.welch(filtered_plc[0], fs=fs_original, nperseg=2048)

plt.figure(figsize=(10, 5))
plt.semilogy(f_raw, Pxx_raw, label='Raw Signal')
plt.semilogy(f_filt, Pxx_filt, label='Filtered Signal (Baseline + 50Hz removed)')
plt.xlabel('Frequency [Hz]')
plt.ylabel('PSD [V**2/Hz]')
plt.title('Power Spectral Density (Welch Periodogram)')
plt.legend()
plt.xlim([0, 100])
plt.show()


## 4. Comparison with ICA (FastICA)

In [ ]:
# ICA operates on the baseline-corrected & PLC-filtered data
n_comp = min(4, filtered_plc.shape[0])
ica = FastICA(n_components=n_comp, random_state=42)
# ICA expects shape (n_samples, n_features)
ica_components = ica.fit_transform(filtered_plc.T)

plt.figure(figsize=(12, 2 * n_comp))
for i in range(n_comp):
    plt.subplot(n_comp, 1, i + 1)
    plt.plot(t_orig[:plot_samples], ica_components[:plot_samples, i])
    plt.title(f'ICA Component {i + 1}')
plt.tight_layout()
plt.show()


## 5. Quality Evaluation (Sample Entropy)

In [ ]:
def sample_entropy(U, m, r):
    """Calculates Sample Entropy."""
    def _maxdist(x_i, x_j):
        return max([abs(ua - va) for ua, va in zip(x_i, x_j)])

    def _phi(m):
        x = [[U[j] for j in range(i, i + m - 1 + 1)] for i in range(N - m + 1)]
        C = [len([1 for j in range(len(x)) if i != j and _maxdist(x[i], x[j]) <= r]) for i in range(len(x))]
        return sum(C)

    N = len(U)
    return -np.log(_phi(m + 1) / _phi(m))

# Calculate SampEn on a window of the residual (downsampled to speed up execution)
eval_signal = signal.resample_poly(residual[:int(5*fs_new)], up=1, down=5)
samp_en = sample_entropy(eval_signal, m=2, r=0.2 * np.std(eval_signal))
print(f'Sample Entropy (SampEn) of Fetal Residual: {samp_en:.4f}')
